## Import testi

Amaç: `src/cdc_generator.py`'yi notebook'tan import edebildiğimizi doğrulamak.

Okuma: `cwd` notebook klasörü (`.../notebooks`) olmalı, `src path` repo kökündeki `src` olmalı, `LANDING_PATH` değeri ekrana basılmalı.

In [0]:
import os
import sys
import importlib

notebook_dir = os.getcwd()
repo_root = os.path.dirname(notebook_dir)
src_path = os.path.join(repo_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

import cdc_generator
importlib.reload(cdc_generator)

print("cwd:", notebook_dir)
print("src path:", src_path)
print("LANDING_PATH:", cdc_generator.LANDING_PATH)

## Parametreler

`NUM_BATCHES`: üretilecek batch sayısı. `SEED`: random seed; aynı değerle aynı dosyalar üretilir.

In [0]:
NUM_BATCHES = 3
SEED = 42

## Güncel durumun okunması (test)

Amaç: `load_current_state` bronze'dan 6 tabloyu Python dictionary'lerine alıyor mu?

Okuma: kayıt sayıları bronze ile aynı olmalı (client 5369, account 4500, card 892, loan 682, district 77, disp 5369). Örnek satırda metadata kolonları olmamalı.

In [0]:
state = cdc_generator.load_current_state(spark)

for table_name, table_state in state.items():
    print(table_name, len(table_state))

print("Sample client:", state["client"]["1"])

## Client değişiklikleri (test, yazma yok)

Amaç: `generate_client_changes` batch 1 için adres değişikliği (U) ve yeni müşteri (I) üretiyor mu?

Okuma: toplam 6–25 kayıt. U'larda `district_id` bronze'dakinden farklı olmalı. I'larda `client_id` 13998'in üstünde olmalı. `change_ts` 1999-01 içinde olmalı. Son satırda bronze değeri değişmemiş olmalı.

In [0]:
import random

rng = random.Random(SEED)
client_changes = cdc_generator.generate_client_changes(state, 1, rng)

print("Total changes:", len(client_changes))
for change in client_changes:
    print(change)

first_id = client_changes[0]["client_id"]
print("Bronze (should be unchanged):", state["client"][first_id])

## Account ve card değişiklikleri (test, yazma yok)

Amaç: `generate_account_changes` ve `generate_card_changes` batch 1 için doğru kayıtları üretiyor mu?

Okuma: Account'ta 5–20 U var, `frequency` bronze'dakinden farklı. Card'da U'lar `gold`, D'ler kartın bronze'daki hali, I'larda `card_id` max'ın üstünde, `issued` `YYMMDD 00:00:00` formatında ve `change_ts` ile aynı gün.

In [0]:
rng = random.Random(SEED)
account_changes = cdc_generator.generate_account_changes(state, 1, rng)
card_changes = cdc_generator.generate_card_changes(state, 1, rng)

print("Account changes:", len(account_changes))
for change in account_changes[:3]:
    print(change)

first_account_id = account_changes[0]["account_id"]
print("Bronze account (should be unchanged):", state["account"][first_account_id])

print()
print("Card changes:", len(card_changes))
for change in card_changes:
    print(change)

## Loan değişiklikleri (test, yazma yok)

Amaç: `generate_loan_changes` batch 1 için sadece `C` durumundaki kredileri `A` veya `D` yapıyor mu?

Okuma: 5–15 U kaydı var, `status` değeri `A` ya da `D`, bronze'da aynı kredinin `status`'u `C`. `C` kredi sayısı bilgi amaçlı basılır.

In [0]:
rng = random.Random(SEED)
loan_changes = cdc_generator.generate_loan_changes(state, 1, rng)

running_count = 0
for loan_row in state["loan"].values():
    if loan_row["status"] == "C":
        running_count = running_count + 1
print("Running (C) loans in bronze:", running_count)

print("Loan changes:", len(loan_changes))
for change in loan_changes:
    loan_id = change["loan_id"]
    bronze_status = state["loan"][loan_id]["status"]
    print(loan_id, bronze_status, "->", change["status"], change["change_ts"])

## Batch 1'in state'e uygulanması (test, yazma yok)

Amaç: `apply_to_state` I / U / D kayıtlarını bellekteki state'e doğru uyguluyor mu?

Okuma: client sayısı I kadar artmalı. Card sayısı = 892 − D + I olmalı. Account ve loan sayıları değişmemeli. Silinen kartlar state'te olmamalı (`False`). Loan ve account'taki yeni değerler state'te görünmeli.

Not: Bu hücre state'i değiştirir. Önceki test hücrelerini tekrar çalıştırmadan önce hücre 6 (`load_current_state`) yeniden çalıştırılmalı.

In [0]:
rng = random.Random(SEED)
batch_changes = {
    "client": cdc_generator.generate_client_changes(state, 1, rng),
    "account": cdc_generator.generate_account_changes(state, 1, rng),
    "card": cdc_generator.generate_card_changes(state, 1, rng),
    "loan": cdc_generator.generate_loan_changes(state, 1, rng),
}

for table_name, changes in batch_changes.items():
    count_before = len(state[table_name])
    cdc_generator.apply_to_state(state, table_name, changes)
    count_after = len(state[table_name])
    print(table_name, "changes:", len(changes), "before:", count_before, "after:", count_after)

print()
for change in batch_changes["card"]:
    if change["op_type"] == "D":
        card_id = change["card_id"]
        print("Deleted card", card_id, "still in state:", card_id in state["card"])

first_loan = batch_changes["loan"][0]
loan_id = first_loan["loan_id"]
print("Loan", loan_id, "status in state:", state["loan"][loan_id]["status"], "(expected", first_loan["status"] + ")")

first_account = batch_changes["account"][0]
account_id = first_account["account_id"]
print("Account", account_id, "frequency in state:", state["account"][account_id]["frequency"])

## 3 batch'in uçtan uca denenmesi (yazma yok)

Amaç: Taze state ile 3 batch'i sırayla üretmek (generate → edge cases → apply) ve op sayılarını görmek.

Okuma: Batch 2'de 2 client ve 1 card ID'si "Edge case" olarak basılır; client'ta U sayısı 2 fazla, card'da U sayısı 1 fazla olur. Batch 3'te olmayan card ve loan ID'leri (max + 100000) basılır; card'da D, loan'da U sayısı 1 fazla olur.

In [0]:
state = cdc_generator.load_current_state(spark)
rng = random.Random(SEED)
all_batches = {}

for batch_id in range(1, NUM_BATCHES + 1):
    print("=== Batch", batch_id, "===")
    batch_changes = {
        "client": cdc_generator.generate_client_changes(state, batch_id, rng),
        "account": cdc_generator.generate_account_changes(state, batch_id, rng),
        "card": cdc_generator.generate_card_changes(state, batch_id, rng),
        "loan": cdc_generator.generate_loan_changes(state, batch_id, rng),
    }
    cdc_generator.add_edge_cases(state, batch_id, rng, batch_changes)

    for table_name, changes in batch_changes.items():
        cdc_generator.apply_to_state(state, table_name, changes)

        op_counts = {"I": 0, "U": 0, "D": 0}
        for change in changes:
            op_counts[change["op_type"]] = op_counts[change["op_type"]] + 1
        print(table_name, op_counts)

    all_batches[batch_id] = batch_changes

## Edge case kayıtlarının kontrolü

Amaç: Batch 2'deki çift kayıtların sırasını ve batch 3'teki olmayan key'lerin state'e girmediğini görmek.

Okuma: Client 435 ve 4206 için batch 2'de iki U kaydı var, `change_ts`'ler farklı. Card 1253 için önce I, sonra U (`gold`). State'te bu key'lerin son hali, sonraki kaydın değeri olmalı (batch 3 aynı key'i tekrar değiştirmediyse). Olmayan key'ler state'te `False` olmalı.

In [0]:
# IDs printed by add_edge_cases in the previous cell
same_key_clients = ["435", "4206"]
same_key_card = "1253"
missing_card = "101254"
missing_loan = "107308"

print("Batch 2 - clients moved twice:")
for change in all_batches[2]["client"]:
    if change["client_id"] in same_key_clients:
        print(" ", change["client_id"], change["op_type"], change["district_id"], change["change_ts"])
for client_id in same_key_clients:
    print("  State now:", state["client"][client_id])

print()
print("Batch 2 - new card upgraded:")
for change in all_batches[2]["card"]:
    if change["card_id"] == same_key_card:
        print(" ", change["op_type"], change["type"], change["issued"], change["change_ts"])
print("  State now:", state["card"][same_key_card])

print()
print("Batch 3 - missing keys stay missing:")
print("  card", missing_card, "in state:", missing_card in state["card"])
print("  loan", missing_loan, "in state:", missing_loan in state["loan"])

## Batch'lerin üretilip landing'e yazılması

Amaç: `run` ile `NUM_BATCHES` kadar batch'i üretip `landing/<tablo>/batch_<NNNN>/` altına JSON lines olarak yazmak. Tekrar çalıştırılırsa aynı dosyalar overwrite edilir.

Okuma: 3 batch × 4 tablo = 12 satır yazma çıktısı. Satır sayıları hücre 16'daki op sayılarının toplamıyla aynı olmalı. Edge case ID'leri yine 435, 4206, 1253, 101254, 107308 olmalı. Son kısımda her tablo klasöründe 3 batch klasörü görünmeli.

In [0]:
cdc_generator.run(spark, NUM_BATCHES, SEED)

print()
for table_name in ["client", "account", "card", "loan"]:
    table_path = f"{cdc_generator.LANDING_PATH}/{table_name}/"
    batch_folders = []
    for file_info in dbutils.fs.ls(table_path):
        batch_folders.append(file_info.name)
    print(table_name, batch_folders)

### P2-A4-S3: Landing batch kontrolleri

**1) Tablo / batch / op_type sayıları**

Landing'deki 4 tablonun JSON dosyaları Spark ile okunur ve `landing_dfs` dictionary'sinde tutulur. Sonraki kontrol hücreleri de bu dictionary'yi kullanır.

Çıktıda her satır bir tablo + batch'tir. `I`, `U`, `D` kolonları op sayılarını gösterir; boş hücre o op'un o batch'te olmadığı anlamına gelir. `total` kolonu, generator'ın yazdığı sayılarla karşılaştırılır.

In [0]:
from pyspark.sql import functions as F

CDC_TABLES = ["client", "account", "card", "loan"]

landing_dfs = {}
df_counts = None

for table_name in CDC_TABLES:
    path = f"{cdc_generator.LANDING_PATH}/{table_name}/batch_*/"
    df_table = spark.read.json(path)
    landing_dfs[table_name] = df_table

    df_table_counts = df_table.groupBy("batch_id", "op_type").count()
    df_table_counts = df_table_counts.withColumn("table_name", F.lit(table_name))

    if df_counts is None:
        df_counts = df_table_counts
    else:
        df_counts = df_counts.unionByName(df_table_counts)

df_pivot = df_counts.groupBy("table_name", "batch_id").pivot("op_type", ["I", "U", "D"]).sum("count")
df_pivot = df_pivot.withColumn("total", F.coalesce(F.col("I"), F.lit(0)) + F.coalesce(F.col("U"), F.lit(0)) + F.coalesce(F.col("D"), F.lit(0)))

display(df_pivot.orderBy("table_name", "batch_id"))

**2) `change_ts` aralıkları**

Her tablo ve batch için en erken ve en geç `change_ts` gösterilir. `change_ts` dosyada ISO string; burada timestamp'e çevrilir.

Beklenen: batch 1 → 1999-01, batch 2 → 1999-02, batch 3 → 1999-03. `in_batch_month` kolonu, min ve max'ın ikisi de batch'in ayı içindeyse `true` olur. Ayrıca `null_ts` kolonu timestamp'e çevrilemeyen kayıt sayısını gösterir; 0 olmalı.

In [0]:
df_ts_ranges = None

for table_name in CDC_TABLES:
    df_table = landing_dfs[table_name]
    df_table = df_table.withColumn("change_ts_ts", F.to_timestamp("change_ts"))

    df_table_ranges = df_table.groupBy("batch_id").agg(
        F.min("change_ts_ts").alias("min_ts"),
        F.max("change_ts_ts").alias("max_ts"),
        F.sum(F.col("change_ts_ts").isNull().cast("int")).alias("null_ts"),
    )
    df_table_ranges = df_table_ranges.withColumn("table_name", F.lit(table_name))

    if df_ts_ranges is None:
        df_ts_ranges = df_table_ranges
    else:
        df_ts_ranges = df_ts_ranges.unionByName(df_table_ranges)

min_ok = (F.year("min_ts") == 1999) & (F.month("min_ts") == F.col("batch_id"))
max_ok = (F.year("max_ts") == 1999) & (F.month("max_ts") == F.col("batch_id"))
df_ts_ranges = df_ts_ranges.withColumn("in_batch_month", min_ok & max_ok)

display(df_ts_ranges.select("table_name", "batch_id", "min_ts", "max_ts", "null_ts", "in_batch_month").orderBy("table_name", "batch_id"))

**3) Batch 1'in bronze ile tutarlılığı**

Batch 1 doğrudan bronze üzerinden üretildi. Bu yüzden her kayıt bronze ile key üzerinden join edilir ve değişen kolonun eski (bronze) ve yeni (landing) değeri karşılaştırılır.

Kolonlar:

- `ud_key_missing`: bronze'da olmayan key'e gelen U/D sayısı → **0** olmalı
- `i_key_exists`: bronze'da zaten olan key'e gelen I sayısı → **0** olmalı
- `u_not_changed`: değeri değişmemiş U sayısı → **0** olmalı
- `d_differs`: bronze'daki son halinden farklı gelen D sayısı → **0** olmalı
- `u_old_values` / `u_new_values`: U'larda eski ve yeni değerlerin listesi (card: `classic` → `gold`, loan: `C` → `A`/`D`)

In [0]:
TRACKED_COLUMNS = {
    "client": "district_id",
    "account": "frequency",
    "card": "type",
    "loan": "status",
}

df_bronze_checks = None

for table_name in CDC_TABLES:
    key_col = cdc_generator.TABLE_KEYS[table_name]
    value_col = TRACKED_COLUMNS[table_name]

    df_bronze = spark.table(f"{cdc_generator.BRONZE_SCHEMA}.{table_name}")
    df_bronze = df_bronze.select(key_col, F.col(value_col).alias("old_value"))

    df_batch1 = landing_dfs[table_name].filter(F.col("batch_id") == 1)
    df_batch1 = df_batch1.select(key_col, "op_type", F.col(value_col).alias("new_value"))

    df_joined = df_batch1.join(df_bronze, on=key_col, how="left")

    is_u = F.col("op_type") == "U"
    is_d = F.col("op_type") == "D"
    is_i = F.col("op_type") == "I"
    key_missing = F.col("old_value").isNull()

    df_check = df_joined.agg(
        F.count("*").alias("rows"),
        F.count_if((is_u | is_d) & key_missing).alias("ud_key_missing"),
        F.count_if(is_i & ~key_missing).alias("i_key_exists"),
        F.count_if(is_u & (F.col("new_value") == F.col("old_value"))).alias("u_not_changed"),
        F.count_if(is_d & (F.col("new_value") != F.col("old_value"))).alias("d_differs"),
        F.array_sort(F.collect_set(F.when(is_u, F.col("old_value")))).alias("u_old_values"),
        F.array_sort(F.collect_set(F.when(is_u, F.col("new_value")))).alias("u_new_values"),
    )
    df_check = df_check.withColumn("table_name", F.lit(table_name))

    if df_bronze_checks is None:
        df_bronze_checks = df_check
    else:
        df_bronze_checks = df_bronze_checks.unionByName(df_check)

display(df_bronze_checks.select("table_name", "rows", "ud_key_missing", "i_key_exists", "u_not_changed", "d_differs", "u_old_values", "u_new_values"))

**4) Edge case kayıtları (seed 42)**

Generator'ın bastığı edge case ID'leri landing'de aranır. `in_bronze` kolonu, key'in bronze'da olup olmadığını gösterir.

Beklenen:

- client 435 ve 4206: batch 2'de iki U, ikincisi daha geç `change_ts` ile (U → U)
- card 1253: batch 2'de önce I `classic`, sonra U `gold` (I → U); bronze'da yok (yeni kart)
- card 101254 (D) ve loan 107308 (U): batch 3'te; bronze'da yok (olmayan key)

In [0]:
EDGE_CASE_IDS = {
    "client": ["435", "4206"],
    "card": ["1253", "101254"],
    "loan": ["107308"],
}

df_edge_cases = None

for table_name in EDGE_CASE_IDS:
    key_col = cdc_generator.TABLE_KEYS[table_name]
    value_col = TRACKED_COLUMNS[table_name]
    edge_ids = EDGE_CASE_IDS[table_name]

    df_bronze_keys = spark.table(f"{cdc_generator.BRONZE_SCHEMA}.{table_name}").select(key_col)
    df_bronze_keys = df_bronze_keys.withColumn("in_bronze", F.lit(True))

    df_edge = landing_dfs[table_name].filter(F.col(key_col).isin(edge_ids))
    df_edge = df_edge.join(df_bronze_keys, on=key_col, how="left")
    df_edge = df_edge.select(
        F.lit(table_name).alias("table_name"),
        F.col(key_col).alias("id"),
        "batch_id",
        "op_type",
        "change_ts",
        F.col(value_col).alias("value"),
        F.coalesce(F.col("in_bronze"), F.lit(False)).alias("in_bronze"),
    )

    if df_edge_cases is None:
        df_edge_cases = df_edge
    else:
        df_edge_cases = df_edge_cases.unionByName(df_edge)

display(df_edge_cases.orderBy("table_name", "id", "batch_id", "change_ts"))